# A Beginner's Introduction to Deep Q-Networks (DQN)

This notebook is a hands-on demo of how a DQN learns to play a simple game. It assumes you know basic Python (variables, loops, functions, classes) but nothing about machine learning or reinforcement learning yet.

## The big idea

Reinforcement learning (RL) is about training an **agent** to make good decisions inside an **environment**, purely from trial and error.

- **State**: what the agent currently observes about the world (numbers describing the situation).
- **Action**: something the agent can do (e.g. move left, move right).
- **Reward**: a number the environment hands back after each action, telling the agent how good that action was.

The agent's goal is simple to state, hard to do: **pick actions that maximize the total reward it collects over time.**

A **DQN (Deep Q-Network)** is one way to build such an agent. The core idea is a function called **Q(state, action)**, which estimates: *"If I'm in this state and I take this action (then play well afterwards), how much total future reward can I expect?"*

If we knew Q perfectly, playing optimally would just mean: in any state, pick the action with the highest Q value. The trick is we don't know Q in advance -- we have to learn it. In a DQN, we use a neural network to approximate Q, and we improve that network's guesses as the agent plays more games.

We'll build this up piece by piece:
1. The game (environment) we'll use
2. A random agent, as a baseline
3. The Q-network (a small neural net)
4. Two key DQN tricks: **experience replay** and a **target network**
5. The training loop, with epsilon-greedy exploration
6. Watching the trained agent improve over time

In [1]:
# Standard imports. Nothing DQN-specific here yet.
import random
from collections import deque
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Makes runs repeatable so the notebook behaves the same way each time.
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. The game: CartPole

We'll use **CartPole**, a classic and very small RL benchmark. A pole is balanced on top of a cart that can move left or right. Gravity constantly tries to topple the pole; the agent's job is to keep it upright as long as possible by nudging the cart.

- **State**: 4 numbers -- cart position, cart velocity, pole angle, pole angular velocity.
- **Actions**: 2 choices -- push the cart left (0) or push it right (1).
- **Reward**: +1 for every timestep the pole stays upright. An episode ends when the pole falls too far, the cart drives off screen, or 500 steps pass.

So the *total reward* for an episode is just "how many steps did the pole stay balanced." A perfect agent scores 500.

In [4]:
env = gym.make("CartPole-v1")

state, info = env.reset(seed=SEED)
print("Example state:", state)
print("Number of actions:", env.action_space.n)

Example state: [ 0.01369617 -0.02302133 -0.04590265 -0.04834723]
Number of actions: 2


## 2. Baseline: an agent that acts randomly

Before training anything, let's see how well an agent does if it just picks random actions. This gives us a number to beat -- if our DQN can't do better than this, something is wrong.

In [5]:
def run_random_agent(num_episodes=20):
    scores = []
    for _ in range(num_episodes):
        state, info = env.reset()
        done = False
        total_reward = 0
        while not done:
            action = env.action_space.sample()  # a completely random action
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
        scores.append(total_reward)
    return scores

random_scores = run_random_agent()
print("Random agent scores:", random_scores)
print("Average:", np.mean(random_scores))

Random agent scores: [12.0, 22.0, 15.0, 15.0, 59.0, 14.0, 22.0, 13.0, 16.0, 12.0, 45.0, 13.0, 9.0, 34.0, 13.0, 23.0, 19.0, 10.0, 45.0, 21.0]
Average: 21.6


Random play usually balances the pole for only ~20-30 steps before it falls. That's our baseline.

## 3. The Q-network

We need a function that takes a state (4 numbers) and outputs a Q value for *each* possible action (2 numbers: "how good is pushing left", "how good is pushing right"). We'll approximate this function with a small neural network.

You don't need to know the details of neural networks to follow this: think of it as a black box with adjustable internal dials (its "weights"). Given a state, it outputs 2 numbers. Training means slowly turning those dials so the outputs become more accurate predictions of future reward.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2),  # one output per action = one Q value per action
        )

    def forward(self, state):
        return self.layers(state)


state_size = env.observation_space.shape[0]  # 4 for CartPole
action_size = env.action_space.n             # 2 for CartPole

test_net = QNetwork(state_size, action_size)
example_input = torch.tensor(state, dtype=torch.float32)
print("Q values for the example state:", test_net(example_input))

Q values for the example state: tensor([0.1368, 0.0020], grad_fn=<ViewBackward0>)


That's an untrained network, so its Q values are meaningless right now -- just random noise. Training is what turns them into useful predictions.

## 4. Two tricks that make DQN work

Plugging a raw neural network directly into Q-learning turns out to be unstable. DQN adds two ideas to fix that.

### Trick 1: Experience replay

Instead of learning only from the most recent step, the agent stores every `(state, action, reward, next_state, done)` it experiences in a memory buffer. During training, it samples a random *batch* of past experiences and learns from those.

Why this helps: consecutive steps in one episode are very similar to each other (the cart barely moves between one frame and the next). Learning from a random mix of past experiences, rather than a stream of near-identical ones, gives the network more varied, less correlated data -- which makes training much more stable.

In [2]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)  # oldest experiences get dropped once full

    def push(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32),
            torch.tensor(actions, dtype=torch.int64),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(np.array(next_states), dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32),
        )

    def __len__(self):
        return len(self.memory)

### Trick 2: A target network

Q-learning trains the network towards a target that is itself computed from the network's own predictions of the *next* state. If we used the exact same, constantly-changing network for both "the prediction" and "the target it's chasing", training becomes like chasing a moving target that dodges every time you get close -- very unstable.

The fix: keep a second copy of the network, called the **target network**. It's used only to compute targets, and its weights are only occasionally updated to match the main network (we'll copy them over every so many training steps). This keeps the target steady for a while, giving the main network something stable to learn towards.

In [3]:
policy_net = QNetwork(state_size, action_size)  # the network we actively train
target_net = QNetwork(state_size, action_size)  # a slow-moving copy, used only for targets
target_net.load_state_dict(policy_net.state_dict())  # start identical
target_net.eval()  # we never train this one directly

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
replay_buffer = ReplayBuffer(capacity=10_000)

print("Networks and replay buffer ready.")

NameError: name 'QNetwork' is not defined

## 5. Choosing actions: epsilon-greedy exploration

If the agent always picked the action its network currently rates highest, it could get stuck: an action it never tries can never be discovered to be good. So we use **epsilon-greedy**:

- With probability `epsilon`, take a completely random action (**explore**).
- Otherwise, take the action the Q-network currently rates highest (**exploit**).

We start with `epsilon` high (mostly random -- the network doesn't know anything useful yet anyway) and decay it over time, so the agent explores a lot early on and increasingly trusts its own judgment as it learns.

In [ ]:
def choose_action(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()  # explore: random action
    with torch.no_grad():
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        q_values = policy_net(state_t)
        return int(torch.argmax(q_values, dim=1).item())  # exploit: best-known action

## 6. The learning step

Here's where the actual math happens. For a batch of past experiences `(state, action, reward, next_state, done)`:

1. Ask the policy network: "what Q value did you predict for the action actually taken?" -- this is our **prediction**.
2. Compute a **target** using the classic Q-learning formula:
   `target = reward + gamma * max(Q_target(next_state))`
   In words: the reward we just got, plus a discounted (`gamma < 1`) estimate of the best we can do from here on, according to the *target* network. If the episode ended (`done`), there is no "afterwards", so the target is just the reward.
3. Nudge the policy network's weights to make its prediction closer to the target (this is what `loss.backward()` + `optimizer.step()` do).

`gamma` (the discount factor) controls how much the agent cares about future rewards vs. immediate ones. `gamma=0.99` means future rewards still matter a lot, just slightly less than immediate ones.

In [ ]:
GAMMA = 0.99
BATCH_SIZE = 64

def train_step():
    if len(replay_buffer) < BATCH_SIZE:
        return  # not enough experience yet to learn from

    states, actions, rewards, next_states, dones = replay_buffer.sample(BATCH_SIZE)

    # Q value the network currently predicts for the action that was actually taken
    predicted_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    # Best Q value available from the next state, according to the (stable) target network
    with torch.no_grad():
        next_q = target_net(next_states).max(dim=1)[0]
        target_q = rewards + GAMMA * next_q * (1 - dones)  # (1 - dones) zeroes out "the future" when the episode ended

    loss = nn.functional.mse_loss(predicted_q, target_q)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

## 7. Putting it all together: the training loop

For each episode:
1. Reset the environment to a fresh starting state.
2. Repeat until the episode ends:
   - Choose an action (epsilon-greedy).
   - Take it, observe the reward and next state.
   - Store the experience in the replay buffer.
   - Run one training step.
3. Decay epsilon a little.
4. Every few episodes, copy the policy network's weights into the target network.

This will take a minute or two to run.

In [1]:
NUM_EPISODES = 300
TARGET_UPDATE_EVERY = 10   # episodes between copying policy_net -> target_net
epsilon = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.98       # multiply epsilon by this after every episode

episode_rewards = []

for episode in range(NUM_EPISODES):
    state, info = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = choose_action(state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        replay_buffer.push(state, action, reward, next_state, done)
        train_step()

        state = next_state
        total_reward += reward

    episode_rewards.append(total_reward)
    epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)

    if episode % TARGET_UPDATE_EVERY == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if episode % 20 == 0:
        recent_avg = np.mean(episode_rewards[-20:])
        print(f"Episode {episode:4d} | reward: {total_reward:5.0f} | avg(last 20): {recent_avg:6.1f} | epsilon: {epsilon:.2f}")

print("Training finished.")

NameError: name 'env' is not defined

## 8. Did it actually learn?

Let's plot the reward per episode. You should see it start low (similar to the random baseline) and trend upward, often noisily, as training progresses.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, label="reward per episode", alpha=0.4)

# a rolling average makes the trend much easier to see through the noise
window = 20
if len(episode_rewards) >= window:
    rolling_avg = np.convolve(episode_rewards, np.ones(window) / window, mode="valid")
    plt.plot(range(window - 1, len(episode_rewards)), rolling_avg, label=f"{window}-episode rolling average", linewidth=2)

plt.axhline(y=np.mean(random_scores), color="red", linestyle="--", label="random-agent average")
plt.xlabel("Episode")
plt.ylabel("Total reward (steps balanced)")
plt.title("DQN learning progress on CartPole")
plt.legend()
plt.show()

## 9. Evaluate the trained agent

Finally, let's run the trained agent with exploration turned off (`epsilon = 0`, always pick the network's best action) and compare its average score to the random baseline from earlier.

In [ ]:
def run_trained_agent(num_episodes=20):
    scores = []
    for _ in range(num_episodes):
        state, info = env.reset()
        done = False
        total_reward = 0
        while not done:
            action = choose_action(state, epsilon=0.0)  # no exploration: trust the network
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
        scores.append(total_reward)
    return scores

trained_scores = run_trained_agent()
print("Trained agent scores:", trained_scores)
print(f"Random agent average:  {np.mean(random_scores):.1f}")
print(f"Trained agent average: {np.mean(trained_scores):.1f}")

env.close()

## Recap

- A DQN learns a function **Q(state, action)** that predicts total future reward, approximated with a neural network.
- **Epsilon-greedy** balances trying new things (exploration) against using what's already been learned (exploitation).
- **Experience replay** stores past experiences and trains on random batches of them, instead of only the most recent step, which stabilizes learning.
- A separate, slowly-updated **target network** provides a stable target for training, instead of chasing a constantly moving prediction.
- Training repeatedly nudges the network's predictions towards `reward + gamma * best future Q`, using real experience gathered by playing the game.

### Ideas to explore next
- Try changing `NUM_EPISODES`, `EPSILON_DECAY`, or the network size (the `64` in `QNetwork`) and see how learning speed changes.
- Try `TARGET_UPDATE_EVERY = 1` (update the target network every episode) and see if training gets less stable.
- Look at `train.py` and `model.py` in this project -- they apply these same ideas to a custom battle game instead of CartPole.